In [6]:
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Annotated, Optional

from IPython.display import Image, display

from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage
from langchain.schema.runnable import RunnableSequence
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

In [2]:
load_dotenv()

True

In [3]:
class State(BaseModel):
    messages: Annotated[list[HumanMessage], Field(
        description="A list of prompts given by the user (human)"
    )]

In [7]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0.6
)
chain = RunnableSequence(llm, StrOutputParser())

In [8]:
def process(state: State) -> State:
    """
    Returns a response corresponding to the prompt sent by the user
    """
    response = chain.invoke(state.messages)
    print(response)
    
    return state
    

In [9]:
graph = StateGraph(State)
graph.add_node("process", process)
graph.set_entry_point("process")
graph.add_edge("process", END)

app = graph.compile()